# Comparaison des quatre variantes de γ

**Modèle de Chiarella étendu** — Majewski, Ciliberti, Bouchaud (2020)

Ce notebook compare les quatre variantes du paramètre de saturation γ sur 1 000 mois (≈ 83 ans) de simulation :

| Variante | Formulation | Paramètre | Effet sur γ |
|----------|-------------|-----------|-------------|
| **Baseline** | $\gamma(t) = \gamma_0$ | — | Constant |
| **Adaptatif linéaire** | $\gamma(t) = \gamma_0 \cdot \left(1 + \theta \cdot \dfrac{\sigma_{\text{réal}}(t)}{\sigma_{\text{baseline}}}\right)$ | $\theta = 1.0$ | ↑ avec volatilité |
| **Mispricing croissant** | $\gamma(t) = \gamma_0 \cdot \exp\!\left(\lambda \cdot \lvert P_t - V_t \rvert\right)$ | $\lambda = 1.0$ | ↑ avec bulle |
| **Mispricing décroissant** | $\gamma(t) = \dfrac{\gamma_0}{1 + \lambda \cdot \lvert P_t - V_t \rvert}$ | $\lambda = 1.0$ | ↓ avec bulle |

**Hypothèse testée** : le modèle décroissant freine explicitement la demande des trend-followers lors des bulles — contrairement aux modèles croissants qui amplifient paradoxalement les signaux intermédiaires.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from src.simulation import (
    run_baseline, run_adaptive, run_mispricing,
    run_decreasing_mispricing, merge_results, VOL_WINDOW_MONTHS,
)
from src.analysis import compute_returns, rolling_volatility, return_statistics, compare_models, gamma_statistics

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11, "legend.fontsize": 10})

# Palette cohérente pour les 4 modèles
COLORS = {
    "baseline":   "steelblue",
    "adaptive":   "darkorange",
    "mispricing": "#8B2FC9",
    "decreasing": "#1A8C5A",
}
LABELS = {
    "baseline":   r"Baseline ($\gamma=\gamma_0$)",
    "adaptive":   r"Adaptatif ($\gamma \propto \sigma_{\text{réal}}$)",
    "mispricing": r"Mispricing↑ ($\gamma \propto e^{\lambda|P-V|}$)",
    "decreasing": r"Mispricing↓ ($\gamma \propto 1/(1+\lambda|P-V|)$)",
}

SEED = 2024
print("Imports OK")

In [ ]:
# ── Simulations ──────────────────────────────────────────────────────────────
print("Simulation baseline …")
res_base, df_base = run_baseline(seed=SEED)

print("Simulation adaptative (θ=1.0) …")
res_adap, df_adap = run_adaptive(theta=1.0, seed=SEED)

print("Simulation mispricing croissant (λ=1.0) …")
res_misp, df_misp = run_mispricing(lambda_=1.0, seed=SEED)

print("Simulation mispricing décroissant (λ=1.0) …")
res_decr, df_decr = run_decreasing_mispricing(lambda_=1.0, seed=SEED)

# Fusion des quatre DataFrames
df = merge_results(df_base, df_adap)
df = pd.merge(df, df_misp, on="t", how="inner")
df = pd.merge(df, df_decr, on="t", how="inner")

# Rendements et mispricing
for label in ("baseline", "adaptive", "mispricing", "decreasing"):
    df[f"{label}_ret"] = df[f"{label}_P"].diff()
    df[f"{label}_mp"]  = df[f"{label}_P"] - df[f"{label}_V"]

# Volatilité réalisée glissante (12 mois)
for label in ("baseline", "adaptive", "mispricing", "decreasing"):
    r = df[f"{label}_ret"].dropna().to_numpy()
    df.loc[df.index[1:], f"{label}_vol"] = rolling_volatility(r, window=VOL_WINDOW_MONTHS)

RESULTS = {"baseline": res_base, "adaptive": res_adap, "mispricing": res_misp, "decreasing": res_decr}

print(f"\nDataFrame : {df.shape[0]} pas × {df.shape[1]} colonnes")
for k, r in RESULTS.items():
    print(f"  γ {k:12s} — mean={r.gamma.mean():.3f}  min={r.gamma.min():.3f}  max={r.gamma.max():.3f}")

---
## Figure 1 — Trajectoires de prix et valeur fondamentale

In [ ]:
t = df["t"].to_numpy()

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08})

ax = axes[0]
for label in ("baseline", "adaptive", "mispricing", "decreasing"):
    ax.plot(t, df[f"{label}_P"], lw=0.8, alpha=0.85,
            color=COLORS[label], label=LABELS[label])
ax.plot(t, df["baseline_V"], lw=1.4, ls="--", color="black", alpha=0.6,
        label=r"$V_t$ (valeur fondamentale)")
ax.set_ylabel("Log-prix / Log-valeur")
ax.set_title("Trajectoires de log-prix — 4 modèles", fontweight="bold")
ax.legend(loc="upper left", fontsize=9)

ax2 = axes[1]
ax2.axhline(0, color="black", lw=1.0, ls="--", alpha=0.5)
for label in ("baseline", "adaptive", "mispricing", "decreasing"):
    ax2.fill_between(t, df[f"{label}_mp"], 0,
                     alpha=0.25, color=COLORS[label], label=LABELS[label])
ax2.set_xlabel("Temps (mois)")
ax2.set_ylabel(r"$P_t - V_t$")
ax2.legend(loc="upper left", fontsize=8, ncol=2)

plt.suptitle(f"Simulation Chiarella étendu — T={int(t[-1])} mois, seed={SEED}",
             y=1.01, fontsize=12, color="gray")
plt.tight_layout()
plt.savefig("fig1_trajectoires.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 2 — Évolution de γ(t) pour les trois modèles dynamiques

- **Adaptatif** : γ ↑ avec σ_réal (proxy VIX)
- **Mispricing↑** : γ ↑ exponentiellement avec |P−V|
- **Mispricing↓** : γ ↓ avec |P−V| — mécanisme de freinage endogène

In [ ]:
t_ret = t[1:]

fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True,
                          gridspec_kw={"hspace": 0.12})

drivers = {
    "adaptive":   ("adaptive_vol",   r"$\sigma_{\text{réal}}(t)$",  "#C0392B"),
    "mispricing": ("mispricing_mp",  r"$|P_t - V_t|$ (croissant↑)", "#5B2A8B"),
    "decreasing": ("decreasing_mp",  r"$|P_t - V_t|$ (décroissant↓)","#0D6E45"),
}

for ax, (label, (driver_col, driver_name, color_driver)) in zip(axes, drivers.items()):
    gamma_series = RESULTS[label].gamma[1:]
    driver_series = np.abs(df[driver_col].to_numpy()[1:]) if "mp" in driver_col else df[driver_col].to_numpy()[1:]

    color_g = COLORS[label]
    ax.plot(t_ret, gamma_series, lw=0.9, alpha=0.9, color=color_g,
            label=rf"$\gamma(t)$ — {LABELS[label]}")
    ax.axhline(2.0, color=color_g, ls=":", lw=1.5, alpha=0.5, label=r"$\gamma_0=2.0$")
    ax.set_ylabel(r"$\gamma(t)$", color=color_g)
    ax.tick_params(axis="y", labelcolor=color_g)
    ax.set_ylim(bottom=0)

    ax2 = ax.twinx()
    ax2.plot(t_ret, driver_series, lw=0.9, alpha=0.55, color=color_driver,
             label=driver_name)
    ax2.set_ylabel(driver_name, color=color_driver)
    ax2.tick_params(axis="y", labelcolor=color_driver)
    ax2.set_ylim(bottom=0)

    mask = ~np.isnan(driver_series)
    rho, p = stats.pearsonr(gamma_series[mask], driver_series[mask])
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labs1 + labs2, loc="upper right", fontsize=8)
    ax.set_title(rf"$\rho(\gamma, \text{{driver}}) = {rho:.4f}$  (p={p:.1e})", fontweight="bold")

axes[-1].set_xlabel("Temps (mois)")
plt.suptitle("Évolution de γ(t) et de ses drivers — 3 modèles dynamiques",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig2_gamma_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 3 — Distribution du mispricing $P_t - V_t$ (4 modèles)

Le modèle **décroissant** est le seul à réduire les bulles extrêmes par rapport au baseline.

In [ ]:
mps = {k: df[f"{k}_mp"].dropna().to_numpy() for k in ("baseline", "adaptive", "mispricing", "decreasing")}
all_mp  = np.concatenate(list(mps.values()))
bins    = np.linspace(all_mp.min(), all_mp.max(), 60)
x_grid  = np.linspace(bins[0], bins[-1], 400)
kdes    = {k: stats.gaussian_kde(m) for k, m in mps.items()}

fig = plt.figure(figsize=(15, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# ── (A) Histogrammes + KDE ────────────────────────────────────────────────────
ax_hist = fig.add_subplot(gs[0, :])
for label, mp in mps.items():
    ax_hist.hist(mp, bins=bins, density=True, alpha=0.20, color=COLORS[label])
    ax_hist.plot(x_grid, kdes[label](x_grid), lw=2, color=COLORS[label], label=LABELS[label])
ax_hist.axvline(0, color="black", ls="--", lw=1.5, alpha=0.5)
ax_hist.set_xlabel(r"Mispricing $P_t - V_t$")
ax_hist.set_ylabel("Densité")
ax_hist.set_title(r"Distribution du mispricing — 4 modèles", fontweight="bold")
ax_hist.legend(fontsize=9)

# ── (B) Queue gauche ─────────────────────────────────────────────────────────
ax_left = fig.add_subplot(gs[1, 0])
q3 = np.percentile(all_mp, 3)
mask_l = x_grid < q3
for label in ("baseline", "adaptive", "mispricing", "decreasing"):
    y = kdes[label](x_grid[mask_l])
    ax_left.plot(x_grid[mask_l], y, lw=2, color=COLORS[label], label=LABELS[label])
    ax_left.fill_between(x_grid[mask_l], y, alpha=0.15, color=COLORS[label])
ax_left.set_xlabel(r"$P_t - V_t$"); ax_left.set_ylabel("Densité (zoom)")
ax_left.set_title("Zoom — queue gauche (3e pct)"); ax_left.legend(fontsize=8)

# ── (C) Queue droite ─────────────────────────────────────────────────────────
ax_right = fig.add_subplot(gs[1, 1])
q97 = np.percentile(all_mp, 97)
mask_r = x_grid > q97
for label in ("baseline", "adaptive", "mispricing", "decreasing"):
    y = kdes[label](x_grid[mask_r])
    ax_right.plot(x_grid[mask_r], y, lw=2, color=COLORS[label], label=LABELS[label])
    ax_right.fill_between(x_grid[mask_r], y, alpha=0.15, color=COLORS[label])
ax_right.set_xlabel(r"$P_t - V_t$"); ax_right.set_ylabel("Densité (zoom)")
ax_right.set_title("Zoom — queue droite (97e pct)"); ax_right.legend(fontsize=8)

plt.suptitle("Analyse des queues de distribution du mispricing",
             fontsize=13, fontweight="bold", y=1.01)
plt.savefig("fig3_mispricing_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Tableau récapitulatif des statistiques

In [ ]:
ref_std = mps["baseline"].std(ddof=1)
ref_max = np.abs(mps["baseline"]).max()

def _mp_stats(mp):
    return {
        "mp_std":     mp.std(ddof=1),
        "mp_max_abs": np.abs(mp).max(),
        "mp_skew":    stats.skew(mp),
        "mp_kurt":    stats.kurtosis(mp, fisher=True),
        "Δstd (%)":   (mp.std(ddof=1) - ref_std) / ref_std * 100,
        "Δmax (%)":   (np.abs(mp).max() - ref_max) / ref_max * 100,
    }

ret_dict = {k: df[f"{k}_ret"].dropna().to_numpy() for k in ("baseline", "adaptive", "mispricing", "decreasing")}
report = compare_models(ret_dict, annualization_factor=np.sqrt(12))

gamma_df = pd.DataFrame({k: gamma_statistics(r.gamma) for k, r in RESULTS.items()}).T
gamma_df.index.name = "model"

mp_df = pd.DataFrame({k: _mp_stats(m) for k, m in mps.items()}).T
mp_df.index.name = "model"

full_stats = pd.concat([report, gamma_df, mp_df], axis=1)
full_stats.index.name = "Modèle"

print("=== Statistiques complètes — 4 modèles ===")
display(full_stats.round(4))

print("\n── Bilan stabilisation des bulles ──")
for label in ("adaptive", "mispricing", "decreasing"):
    d_std = (mps[label].std(ddof=1) - ref_std) / ref_std * 100
    d_max = (np.abs(mps[label]).max() - ref_max) / ref_max * 100
    direction = "✅ réduit" if d_std < 0 else "❌ aggrave"
    print(f"  {label:12s} {direction} | Δstd = {d_std:+.2f} %  Δmax|mp| = {d_max:+.2f} %")